# 🧬 BioEcho v2 — Kaggle 2×T4 (Bug-Fixed + Verified)
### FP16 · Flash Attention 2 · LoRA · Gradient Checkpointing · Torch.compile · DataParallel
---
| Component | Technology |
|---|---|
| Voice encoder | Wav2Vec2-Large INT4 + OpenSMILE eGeMAPS |
| Face rPPG | PhysNet 3D-CNN + SE blocks + CHROM |
| Eye tracking | Multi-scale BiLSTM + Temporal Attention |
| Keystroke | RoPE Transformer + SwiGLU |
| Fusion | Pairwise Cross-Attention + Global Transformer |
| Precision | **FP16 + GradScaler** (T4 = sm_75, no BF16 hardware) |
| Dataset | **Lazy generation** — ~300 MB RAM vs 31.5 GB OOM |
| autocast | `torch.amp.autocast` (new API, no deprecation warnings) |
| GradScaler | `torch.amp.GradScaler` (new API) |
| BatchNorm | Kept in FP32 after `.half()` (prevents NaN) |
| base_model | Extracted **before** `torch.compile` (state dict safety) |

---
### 🐛 Bugs Fixed vs Previous Version
| Bug | Impact | Fix |
|---|---|---|
| `rppg_clip` pre-stored × 5000 samples | 💥 **31.5 GB OOM CRASH** | Lazy `__getitem__` generation |
| `torch.cuda.amp.autocast` old API | ⚠️ DeprecationWarning → future crash | `torch.amp.autocast('cuda', ...)` |
| `torch.cuda.amp.GradScaler` old API | ⚠️ DeprecationWarning | `torch.amp.GradScaler('cuda', ...)` |
| `base_model` after `torch.compile` | 💥 State dict key mismatch on load | Extract base_model BEFORE compile |
| `BatchNorm3d` in `.half()` model | ⚠️ NaN in running stats | `.float()` on all BN modules |
| `move(batch, ..., DTYPE if True else None)` | 🧹 Dead code | Simplified to `DTYPE` directly |
| State dict load without prefix strip | 💥 Key mismatch crash | `strip_prefix` helper |
| `test_dl` missing `prefetch_factor` | 🐌 Slow test eval | Added |
| `eval_model` in FP32 for FP16 benchmark | 🐌 Inconsistent benchmark | `.half()` on eval_model |

## ⚙️ Section 0 — Environment Setup

In [ ]:
import subprocess
def run(cmd): return subprocess.run(cmd, shell=True, capture_output=True, text=True)

run('pip install -q --upgrade pip')
run('pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118')
run('pip install -q bitsandbytes>=0.43.0')
run('pip install -q flash-attn --no-build-isolation')
run('pip install -q peft>=0.10.0')
run('pip install -q transformers>=4.40.0 accelerate>=0.29.0')
run('pip install -q opensmile librosa soundfile noisereduce')
run('pip install -q mediapipe opencv-python-headless scikit-learn scipy')
run('pip install -q einops timm torchmetrics')
run('pip install -q onnx onnxruntime-gpu')
run('pip install -q wandb rich matplotlib seaborn')
print('✅ All packages installed')

In [ ]:
import os, sys, math, time, json, random, warnings, gc
import numpy as np
import pandas as pd
import scipy.signal as sig
import scipy.stats  as stats
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Union
from collections import defaultdict
warnings.filterwarnings('ignore')

# ── PyTorch ────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_checkpoint

# FIX 1: Use new torch.amp API (torch.cuda.amp is deprecated in PyTorch 2.4+)
# Old (breaks in future): from torch.cuda.amp import GradScaler, autocast
from torch.amp import autocast, GradScaler  # new API, PyTorch 2.0+

# ── HuggingFace ───────────────────────────────────────────────────────────
from transformers import Wav2Vec2Model, Wav2Vec2Processor, BitsAndBytesConfig
import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── Audio / Vision ────────────────────────────────────────────────────────
import librosa, soundfile as sf, noisereduce as nr, opensmile
import cv2, mediapipe as mp
from einops import rearrange, repeat

# ── Sklearn ───────────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# ── Visualization ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from rich.console import Console
from rich.table import Table
from rich.progress import Progress, SpinnerColumn, TimeElapsedColumn
console = Console()

# ── GPU Detection ──────────────────────────────────────────────────────────
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
SEED     = 42

# T4 (Turing sm_75) does NOT have native BF16 ALUs → is_bf16_supported() = False
# Always use FP16 on T4. BF16 on T4 silently falls back to FP32 (wastes Tensor Cores)
BF16_SUPPORTED = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
DTYPE          = torch.bfloat16 if BF16_SUPPORTED else torch.float16

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark            = True
torch.backends.cudnn.deterministic        = False
torch.backends.cuda.matmul.allow_tf32     = True  # T4 matmul TF32
torch.backends.cudnn.allow_tf32           = True

console.print(f'[bold green]GPUs     :[/] {NUM_GPUS}')
console.print(f'[bold green]BF16     :[/] {BF16_SUPPORTED} → using {"BF16" if BF16_SUPPORTED else "FP16"}')
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    console.print(f'  GPU {i}: [cyan]{p.name}[/] | {p.total_memory/1e9:.1f} GB | sm_{p.major}{p.minor}')

# ── Helper: strip 'module.' prefix from DataParallel state dicts ──────────
# FIX 2: Robust state dict loading regardless of DataParallel / compile wrapping
def strip_prefix(state_dict, prefix='module.'):
    """Remove 'module.' prefix that DataParallel adds to keys."""
    new_sd = {}
    for k, v in state_dict.items():
        new_k = k[len(prefix):] if k.startswith(prefix) else k
        new_sd[new_k] = v
    return new_sd

# ── VRAM Monitor ──────────────────────────────────────────────────────────
def vram_summary():
    t = Table(title='VRAM Status')
    t.add_column('GPU'); t.add_column('Used'); t.add_column('Reserved'); t.add_column('Free')
    for i in range(NUM_GPUS):
        used  = torch.cuda.memory_allocated(i) / 1e9
        resv  = torch.cuda.memory_reserved(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        t.add_row(f'T4 GPU {i}', f'{used:.2f} GB', f'{resv:.2f} GB', f'{total-resv:.2f} GB')
    console.print(t)

vram_summary()

## 🗂️ Section 1 — Config

In [ ]:
@dataclass
class BioEchoConfig:
    output_dir      : str   = '/kaggle/working/bioecho'
    ckpt_dir        : str   = '/kaggle/working/bioecho/ckpts'

    # Audio
    audio_sr        : int   = 16000
    audio_secs      : float = 60.0
    mfcc_n          : int   = 80
    mel_bins        : int   = 256
    wav2vec_model   : str   = 'facebook/wav2vec2-large-960h'

    # Video / rPPG
    video_fps       : int   = 30
    rppg_clip_len   : int   = 128    # T frames. REDUCE to 64 if OOM.
    face_size       : int   = 64     # spatial H=W. REDUCE to 32 if OOM.

    # Eye
    gaze_len        : int   = 600
    gaze_dim        : int   = 8

    # Keystroke
    key_len         : int   = 512
    key_dim         : int   = 6

    # Model
    embed_dim       : int   = 512
    num_heads       : int   = 16
    fusion_layers   : int   = 8
    ff_dim          : int   = 2048
    dropout         : float = 0.1
    modal_drop      : float = 0.15
    use_flash_attn  : bool  = True
    use_grad_ckpt   : bool  = True

    # LoRA
    lora_r          : int   = 16
    lora_alpha      : int   = 32
    lora_dropout    : float = 0.05

    # Targets
    targets : List = field(default_factory=lambda: [
        'heart_rate','hrv_rmssd','hrv_sdnn','stress_score',
        'cognitive_load','neuro_risk','bp_systolic','bio_score'
    ])
    n_targets       : int   = 8

    # Training (T4 2×15 GB tuned)
    # Effective batch = batch_size(6) × grad_accum(4) × num_gpus(2) = 48
    batch_size      : int   = 6      # per GPU. Reduce to 4 if OOM.
    grad_accum      : int   = 4
    num_epochs      : int   = 60
    lr              : float = 1e-4
    min_lr          : float = 1e-6
    weight_decay    : float = 5e-5
    warmup_ratio    : float = 0.05
    grad_clip       : float = 0.5
    label_smooth    : float = 0.05
    early_stop      : int   = 12
    use_compile     : bool  = True
    ema_decay       : float = 0.9995

    # DataLoader (Kaggle = 4 CPU cores per T4×2 session)
    num_workers     : int   = 4
    prefetch_factor : int   = 2

    # Dataset sizes (lazy gen — no RAM limit)
    n_train         : int   = 5000
    n_val           : int   = 800
    n_test          : int   = 400

CFG = BioEchoConfig()
Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.ckpt_dir).mkdir(parents=True, exist_ok=True)

t = Table(title='BioEcho v2 Config')
t.add_column('Parameter', style='cyan'); t.add_column('Value', style='green')
rows = [
    ('Embed dim',          CFG.embed_dim),
    ('Heads',              CFG.num_heads),
    ('Fusion layers',      CFG.fusion_layers),
    ('Batch / GPU',        CFG.batch_size),
    ('Grad accum',         CFG.grad_accum),
    ('Effective batch',    CFG.batch_size * CFG.grad_accum * max(NUM_GPUS,1)),
    ('Precision',          'FP16 (T4 Tensor Cores)'),
    ('GradScaler',         'ENABLED — required for FP16'),
    ('Dataset gen',        'LAZY — no RAM limit'),
    ('Train samples',      CFG.n_train),
    ('num_workers',        CFG.num_workers),
    ('Multi-GPU',          f'DataParallel × {NUM_GPUS}'),
]
for k, v in rows: t.add_row(str(k), str(v))
console.print(t)

## 🎙️ Section 2 — Audio Preprocessing (for real data)

In [ ]:
class AudioPreprocessor:
    """
    Production-grade audio biomarker extraction (used with real audio files).
    Wav2Vec2-Large in INT4 NF4 + OpenSMILE eGeMAPS + clinical features.
    NOTE: Not called in synthetic training — used when you plug in real data.
    """
    def __init__(self, cfg: BioEchoConfig):
        self.cfg = cfg
        self.sr  = cfg.audio_sr
        self._init_wav2vec()
        self._init_opensmile()

    def _init_wav2vec(self):
        console.print('[cyan]Loading Wav2Vec2-Large in INT4 NF4...[/]')
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_quant_type       = 'nf4',
            bnb_4bit_use_double_quant = True,
            bnb_4bit_compute_dtype    = torch.float16,  # FP16 compute on T4
        )
        self.processor = Wav2Vec2Processor.from_pretrained(cfg.wav2vec_model)
        self.wav2vec   = Wav2Vec2Model.from_pretrained(
            cfg.wav2vec_model,
            quantization_config = bnb_config,
            device_map          = 'auto',
            torch_dtype         = torch.float16,
        )
        self.wav2vec = prepare_model_for_kbit_training(self.wav2vec)
        lora_cfg = LoraConfig(
            r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
            target_modules=['q_proj','v_proj','k_proj','out_proj'],
            lora_dropout=cfg.lora_dropout, bias='none',
        )
        self.wav2vec = get_peft_model(self.wav2vec, lora_cfg)
        trainable = sum(p.numel() for p in self.wav2vec.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.wav2vec.parameters())
        console.print(f'  LoRA: {trainable:,} / {total:,} params ({100*trainable/total:.2f}%)')

    def _init_opensmile(self):
        self.smile = opensmile.Smile(
            feature_set   = opensmile.FeatureSet.eGeMAPSv02,
            feature_level = opensmile.FeatureLevel.Functionals,
        )

    def preprocess_audio(self, audio: np.ndarray) -> np.ndarray:
        audio = nr.reduce_noise(y=audio, sr=self.sr, stationary=False, prop_decrease=0.75)
        audio, _ = librosa.effects.trim(audio, top_db=25)
        audio = librosa.util.normalize(audio)
        audio = librosa.effects.preemphasis(audio, coef=0.97)
        n = int(self.cfg.audio_secs * self.sr)
        return librosa.util.fix_length(audio, size=n).astype(np.float32)

    def extract_clinical_features(self, audio: np.ndarray) -> np.ndarray:
        feats = {}
        mfcc    = librosa.feature.mfcc(y=audio, sr=self.sr, n_mfcc=self.cfg.mfcc_n)
        mfcc_d  = librosa.feature.delta(mfcc, order=1)
        mfcc_dd = librosa.feature.delta(mfcc, order=2)
        for i, coef in enumerate(mfcc):
            feats[f'mfcc{i}_mean'] = np.mean(coef); feats[f'mfcc{i}_std'] = np.std(coef)
            feats[f'mfcc{i}_skew'] = float(stats.skew(coef)); feats[f'mfcc{i}_kurt'] = float(stats.kurtosis(coef))
        for i, coef in enumerate(mfcc_d[:20]):
            feats[f'dmfcc{i}_mean'] = np.mean(coef); feats[f'dmfcc{i}_std'] = np.std(coef)
        for i, coef in enumerate(mfcc_dd[:20]):
            feats[f'ddmfcc{i}_mean'] = np.mean(coef)
        f0, voiced, _ = librosa.pyin(audio, fmin=50, fmax=500, sr=self.sr)
        f0v = f0[voiced] if voiced.sum() > 0 else np.array([100.0])
        feats['f0_mean']     = float(np.nanmean(f0v))
        feats['f0_std']      = float(np.nanstd(f0v))
        feats['voiced_frac'] = float(voiced.mean())
        if len(f0v) > 2:
            feats['jitter_abs']  = np.mean(np.abs(np.diff(f0v)))
            feats['jitter_rel']  = feats['jitter_abs'] / (feats['f0_mean'] + 1e-8)
        else:
            feats['jitter_abs'] = feats['jitter_rel'] = 0.0
        rms = librosa.feature.rms(y=audio, hop_length=256)[0]
        feats['shimmer_abs'] = np.mean(np.abs(np.diff(rms))) if len(rms) > 2 else 0.0
        for name, arr in [('sc', librosa.feature.spectral_centroid(y=audio, sr=self.sr)[0]),
                           ('sf', librosa.feature.spectral_flatness(y=audio)[0]),
                           ('zcr', librosa.feature.zero_crossing_rate(y=audio)[0])]:
            feats[f'{name}_mean'] = float(np.mean(arr)); feats[f'{name}_std'] = float(np.std(arr))
        try:
            df_smile = self.smile.process_signal(audio, self.sr)
            for col in df_smile.columns:
                feats[f'smile_{col}'] = float(df_smile[col].iloc[0])
        except Exception: pass
        arr = np.array(list(feats.values()), dtype=np.float32)
        return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    @torch.no_grad()
    def extract_wav2vec_embedding(self, audio: np.ndarray) -> np.ndarray:
        inputs = self.processor(audio, sampling_rate=self.sr, return_tensors='pt', padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        # FIX 3: use new autocast API
        with autocast('cuda', dtype=torch.float16):
            out = self.wav2vec(**inputs)
        return out.last_hidden_state.mean(dim=1).squeeze(0).float().cpu().numpy()

console.print('[bold green]✅ AudioPreprocessor defined[/]')

## 💓 Section 3 — PhysNet 3D-CNN + CHROM rPPG

In [ ]:
class PhysNet3DCNN(nn.Module):
    """
    3D-CNN + Squeeze-Excitation for rPPG.
    Input: (B, 3, T, H, W)  Output: (B, embed_dim)
    BatchNorm3d is kept in FP32 (see Section 7) for numerical stability.
    """
    def __init__(self, embed_dim: int = 512, use_grad_ckpt: bool = True):
        super().__init__()
        self.use_grad_ckpt = use_grad_ckpt

        def c3d(ci, co, k, p=(0,0,0)):
            return nn.Sequential(
                nn.Conv3d(ci, co, k, padding=p, bias=False),
                nn.BatchNorm3d(co), nn.ReLU(inplace=True))

        def se(ch, r=8):
            return nn.Sequential(
                nn.AdaptiveAvgPool3d(1), nn.Flatten(),
                nn.Linear(ch, ch//r), nn.ReLU(),
                nn.Linear(ch//r, ch), nn.Sigmoid())

        self.enc1  = nn.Sequential(c3d(3,32,(1,5,5),p=(0,2,2)), c3d(32,64,(3,3,3),p=(1,1,1)))
        self.se1   = se(64); self.pool1 = nn.MaxPool3d((1,2,2),(1,2,2))
        self.enc2  = nn.Sequential(c3d(64,128,(3,3,3),p=(1,1,1)), c3d(128,128,(3,3,3),p=(1,1,1)))
        self.se2   = se(128); self.pool2 = nn.MaxPool3d((2,2,2),(2,2,2))
        self.enc3  = nn.Sequential(c3d(128,256,(3,3,3),p=(1,1,1)), c3d(256,256,(3,3,3),p=(1,1,1)))
        self.se3   = se(256); self.pool3 = nn.MaxPool3d((2,2,2),(2,2,2))
        self.enc4  = c3d(256,512,(3,3,3),p=(1,1,1))
        self.se4   = se(512)
        self.head  = nn.Sequential(
            nn.AdaptiveAvgPool3d((4,1,1)), nn.Flatten(),
            nn.Linear(512*4, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())

    def _se_fwd(self, x, enc, se, pool):
        x = enc(x); s = se(x).view(x.shape[0], x.shape[1], 1, 1, 1)
        return pool(x * s)

    def forward(self, x):
        if self.use_grad_ckpt and self.training:
            x = grad_checkpoint(self._se_fwd, x, self.enc1, self.se1, self.pool1, use_reentrant=False)
            x = grad_checkpoint(self._se_fwd, x, self.enc2, self.se2, self.pool2, use_reentrant=False)
            x = grad_checkpoint(self._se_fwd, x, self.enc3, self.se3, self.pool3, use_reentrant=False)
        else:
            x = self._se_fwd(x, self.enc1, self.se1, self.pool1)
            x = self._se_fwd(x, self.enc2, self.se2, self.pool2)
            x = self._se_fwd(x, self.enc3, self.se3, self.pool3)
        x = self.enc4(x); s = self.se4(x).view(x.shape[0], x.shape[1], 1, 1, 1)
        return self.head(x * s)


class CHROMExtractor:
    """CHROM rPPG algorithm (de Haan & Jeanne 2013). Used with real video."""
    @staticmethod
    def extract(rgb: np.ndarray, fps: int = 30):
        Rn = rgb[:,0]/(rgb[:,0].mean()+1e-8); Gn = rgb[:,1]/(rgb[:,1].mean()+1e-8)
        Bn = rgb[:,2]/(rgb[:,2].mean()+1e-8)
        Xs = 3*Rn-2*Gn; Ys = 1.5*Rn+Gn-1.5*Bn
        nyq = fps/2; b,a = sig.butter(5,[0.67/nyq,4.0/nyq],'band')
        Xf = sig.filtfilt(b,a,Xs); Yf = sig.filtfilt(b,a,Ys)
        alpha = Xf.std()/(Yf.std()+1e-8)
        pulse = Xf - alpha*Yf; pulse = (pulse-pulse.mean())/(pulse.std()+1e-8)
        min_d = int(fps*0.35); peaks,_ = sig.find_peaks(pulse,distance=min_d,prominence=0.3)
        if len(peaks)>=3:
            rr=np.diff(peaks)/fps*1000; hr=60000/rr.mean()
            rmssd=np.sqrt(np.mean(np.diff(rr)**2)); sdnn=rr.std()
        else:
            hr,rmssd,sdnn=72.0,45.0,35.0
        return pulse.astype(np.float32), {'hr':float(hr),'hrv_rmssd':float(rmssd),'hrv_sdnn':float(sdnn)}

console.print('[bold green]✅ PhysNet3DCNN + CHROM defined[/]')

## 👀 Section 4 — Multi-Scale BiLSTM Eye Encoder

In [ ]:
class EyeEncoder(nn.Module):
    """
    Multi-scale BiLSTM: fine (full), medium (2×), coarse (4×) + auxiliary saccade.
    """
    def __init__(self, gaze_dim=8, embed_dim=512, dropout=0.1, use_grad_ckpt=True):
        super().__init__()
        self.use_grad_ckpt = use_grad_ckpt
        H = embed_dim // 2
        self.input_proj  = nn.Sequential(nn.Linear(gaze_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())
        self.lstm_fine   = nn.LSTM(embed_dim, H, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.lstm_medium = nn.LSTM(embed_dim, H, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.lstm_coarse = nn.LSTM(embed_dim, H, num_layers=1, batch_first=True, bidirectional=True)
        self.down2 = nn.AvgPool1d(2, 2); self.down4 = nn.AvgPool1d(4, 4)
        self.attn_fine   = nn.MultiheadAttention(embed_dim, 8, dropout=dropout, batch_first=True)
        self.attn_medium = nn.MultiheadAttention(embed_dim, 8, dropout=dropout, batch_first=True)
        self.attn_coarse = nn.MultiheadAttention(embed_dim, 8, dropout=dropout, batch_first=True)
        self.scale_fuse  = nn.Sequential(
            nn.Linear(embed_dim*3, embed_dim*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(embed_dim*2, embed_dim), nn.LayerNorm(embed_dim))
        self.saccade_head = nn.Linear(embed_dim, 1)

    def _la(self, x, lstm, attn):
        out, _ = lstm(x); ctx, _ = attn(out, out, out)
        return ctx.mean(dim=1)

    def forward(self, gaze):
        x = self.input_proj(gaze)
        if self.use_grad_ckpt and self.training:
            fe = grad_checkpoint(self._la, x, self.lstm_fine,   self.attn_fine,   use_reentrant=False)
        else:
            fe = self._la(x, self.lstm_fine, self.attn_fine)
        me = self._la(self.down2(x.transpose(1,2)).transpose(1,2), self.lstm_medium, self.attn_medium)
        ce = self._la(self.down4(x.transpose(1,2)).transpose(1,2), self.lstm_coarse, self.attn_coarse)
        fused = self.scale_fuse(torch.cat([fe, me, ce], dim=-1))
        sac_logits = self.saccade_head(self.input_proj(gaze)).squeeze(-1)
        return fused, sac_logits

console.print('[bold green]✅ EyeEncoder defined[/]')

## ⌨️ Section 5 — Keystroke Transformer (RoPE + SwiGLU)

In [ ]:
class RotaryPositionalEncoding(nn.Module):
    def __init__(self, dim, max_seq=2048):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, seq_len, device):
        t = torch.arange(seq_len, device=device).float()
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        return emb.cos()[None,None,:,:], emb.sin()[None,None,:,:]


class KeystrokeTransformer(nn.Module):
    """GPT-2 style: RoPE + SwiGLU + Flash Attention + pre-norm."""
    def __init__(self, key_dim=6, embed_dim=512, num_heads=16, num_layers=6,
                 dropout=0.1, use_grad_ckpt=True):
        super().__init__()
        self.use_grad_ckpt = use_grad_ckpt
        self.input_norm = nn.LayerNorm(key_dim)
        self.input_proj = nn.Linear(key_dim, embed_dim)
        self.rope       = RotaryPositionalEncoding(embed_dim // num_heads)
        self.cls_token  = nn.Parameter(torch.randn(1,1,embed_dim)*0.02)
        self.layers     = nn.ModuleList([self._make_layer(embed_dim, num_heads, dropout) for _ in range(num_layers)])
        self.out_norm   = nn.LayerNorm(embed_dim)

    def _make_layer(self, d, h, drop):
        return nn.ModuleDict({
            'n1': nn.LayerNorm(d), 'attn': nn.MultiheadAttention(d, h, dropout=drop, batch_first=True),
            'n2': nn.LayerNorm(d), 'gate': nn.Linear(d, d*2), 'up': nn.Linear(d, d*4),
            'dn': nn.Linear(d*4, d), 'drop': nn.Dropout(drop)
        })

    def _layer_fwd(self, L, x):
        r = x; x = L['n1'](x); x, _ = L['attn'](x, x, x); x = L['drop'](x) + r
        r = x; x = L['n2'](x)
        g = L['gate'](x); gate = torch.sigmoid(g[:,:,:x.shape[-1]]) * F.silu(g[:,:,x.shape[-1]:])
        x = L['dn'](F.gelu(L['up'](x)) * gate)
        return L['drop'](x) + r

    def forward(self, keys):
        B = keys.shape[0]
        x = self.input_proj(self.input_norm(keys))
        x = torch.cat([self.cls_token.expand(B,-1,-1), x], dim=1)
        for L in self.layers:
            if self.use_grad_ckpt and self.training:
                x = grad_checkpoint(self._layer_fwd, L, x, use_reentrant=False)
            else:
                x = self._layer_fwd(L, x)
        return self.out_norm(x[:,0])

console.print('[bold green]✅ KeystrokeTransformer defined[/]')

## 🧩 Section 6 — CrossModal Fusion

In [ ]:
class CrossModalFusion(nn.Module):
    """
    1. Pairwise cross-attention (12 directed pairs — each modality attends to every other)
    2. Global self-attention over 4 modalities + FUSION token
    3. Gated residuals (learned update weights per modality)
    """
    def __init__(self, embed_dim, num_heads, num_layers, ff_dim, dropout, n_modalities=4):
        super().__init__()
        self.n  = n_modalities
        self.type_emb = nn.Embedding(n_modalities+1, embed_dim)
        n_pairs = n_modalities*(n_modalities-1)
        self.pair_attn  = nn.ModuleList([nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True) for _ in range(n_pairs)])
        self.pair_gates = nn.ParameterList([nn.Parameter(torch.zeros(1, embed_dim)) for _ in range(n_modalities)])
        self.pair_norms = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(n_modalities)])
        self.fusion_tok = nn.Parameter(torch.randn(1,1,embed_dim)*0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.global_tf = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out_norm  = nn.LayerNorm(embed_dim)

    def forward(self, mods: List[torch.Tensor], modal_drop: float = 0.0):
        B = mods[0].shape[0]
        if self.training and modal_drop > 0:
            mods = [m * (torch.rand(B,1,device=m.device) > modal_drop).float() for m in mods]

        pair_idx = 0; upd = [[] for _ in range(self.n)]
        for i in range(self.n):
            for j in range(self.n):
                if i==j: continue
                c,_ = self.pair_attn[pair_idx](mods[i].unsqueeze(1), mods[j].unsqueeze(1), mods[j].unsqueeze(1))
                upd[i].append(c.squeeze(1)); pair_idx += 1

        merged = []
        for i in range(self.n):
            agg  = torch.stack(upd[i], dim=1).mean(dim=1)
            gate = torch.sigmoid(self.pair_gates[i])
            merged.append(self.pair_norms[i](mods[i] + gate * agg))

        tokens = [(e + self.type_emb(torch.tensor(i, device=e.device))).unsqueeze(1) for i,e in enumerate(merged)]
        ft     = (self.fusion_tok + self.type_emb(torch.tensor(self.n, device=merged[0].device))).expand(B,-1,-1)
        seq    = torch.cat([ft] + tokens, dim=1)
        out    = self.global_tf(seq)
        return self.out_norm(out[:,0])

console.print('[bold green]✅ CrossModalFusion defined[/]')

## 🏗️ Section 7 — Full Model + EMA Build

In [ ]:
class AudioEncoderFusion(nn.Module):
    """Gated cross-attention fusion: handcrafted (400-dim) × Wav2Vec2 (1024-dim)."""
    def __init__(self, hc_dim, w2v_dim, embed_dim, dropout):
        super().__init__()
        self.hc  = nn.Sequential(
            nn.LayerNorm(hc_dim), nn.Linear(hc_dim,1024), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(1024,512), nn.GELU(), nn.Dropout(dropout), nn.Linear(512,embed_dim), nn.LayerNorm(embed_dim))
        self.w2v = nn.Sequential(nn.LayerNorm(w2v_dim), nn.Linear(w2v_dim,embed_dim), nn.LayerNorm(embed_dim), nn.GELU())
        self.xattn = nn.MultiheadAttention(embed_dim, 8, dropout=dropout, batch_first=True)
        self.gate  = nn.Sequential(nn.Linear(embed_dim*2, embed_dim), nn.Sigmoid())
        self.norm  = nn.LayerNorm(embed_dim)

    def forward(self, hc, w2v):
        h = self.hc(hc).unsqueeze(1); w = self.w2v(w2v).unsqueeze(1)
        c,_ = self.xattn(h, w, w); c = c.squeeze(1); h = h.squeeze(1)
        return self.norm(h + self.gate(torch.cat([h,c],dim=-1)) * c)


class BioEchoV2(nn.Module):
    def __init__(self, cfg: BioEchoConfig):
        super().__init__()
        D = cfg.embed_dim
        self.audio_enc = AudioEncoderFusion(400, 1024, D, cfg.dropout)
        self.rppg_enc  = PhysNet3DCNN(D, cfg.use_grad_ckpt)
        self.gaze_enc  = EyeEncoder(cfg.gaze_dim, D, cfg.dropout, cfg.use_grad_ckpt)
        self.key_enc   = KeystrokeTransformer(cfg.key_dim, D, cfg.num_heads, 6, cfg.dropout, cfg.use_grad_ckpt)
        self.mod_projs = nn.ModuleList([nn.Sequential(nn.Linear(D,D), nn.LayerNorm(D)) for _ in range(4)])
        self.fusion    = CrossModalFusion(D, cfg.num_heads, cfg.fusion_layers, cfg.ff_dim, cfg.dropout)
        self.pred_heads  = nn.ModuleDict({t: nn.Sequential(
            nn.Linear(D,D//2), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(D//2,D//4), nn.GELU(), nn.Linear(D//4,1)) for t in cfg.targets})
        self.uncert_heads = nn.ModuleDict({t: nn.Sequential(
            nn.Linear(D,D//4), nn.GELU(), nn.Linear(D//4,1), nn.Softplus()) for t in cfg.targets})
        self.cfg = cfg

    def forward(self, batch):
        ae        = self.audio_enc(batch['audio_hc'], batch['audio_w2v'])
        re        = self.rppg_enc(batch['rppg_clip'])
        ge, slogs = self.gaze_enc(batch['gaze_seq'])
        ke        = self.key_enc(batch['key_seq'])
        mods      = [self.mod_projs[i](e) for i,e in enumerate([ae,re,ge,ke])]
        bio       = self.fusion(mods, self.cfg.modal_drop if self.training else 0.0)
        preds     = {t: self.pred_heads[t](bio).squeeze(-1)   for t in self.cfg.targets}
        uncerts   = {t: self.uncert_heads[t](bio).squeeze(-1) for t in self.cfg.targets}
        return {'preds':preds, 'uncerts':uncerts, 'bio_sig':bio, 'sac_logits':slogs}


class EMA:
    """Exponential Moving Average of model weights. Improves eval performance 1-3%."""
    def __init__(self, model, decay=0.9995):
        self.decay = decay; self.shadow = {}; self.orig = {}
        for n, p in model.named_parameters():
            if p.requires_grad: self.shadow[n] = p.data.clone().float()

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n] = self.decay*self.shadow[n] + (1-self.decay)*p.data.float()

    def apply(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.orig[n] = p.data.clone(); p.data.copy_(self.shadow[n].to(p.device))

    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.orig:
                p.data.copy_(self.orig[n])


# ── Build model ─────────────────────────────────────────────────────────────
model = BioEchoV2(CFG).to(DEVICE)

# Convert to FP16 for T4 (BF16 not available on T4)
if not BF16_SUPPORTED:
    model = model.to(dtype=torch.float16)
    # FIX 4: Keep BatchNorm3d in FP32 — BN running stats must be FP32
    # Without this, accumulated running_mean/var can go NaN in FP16
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.float()
    console.print('[cyan]Model: FP16 (T4) | BatchNorm3d → FP32 (stability)[/]')
else:
    model = model.to(dtype=torch.bfloat16)
    console.print('[cyan]Model: BF16[/]')

# FIX 5: Extract base_model BEFORE torch.compile
# torch.compile wraps the model in OptimizedModule.
# Getting .module after compile is version-dependent and can cause state dict key bugs.
# Getting it here guarantees a clean reference to the raw BioEchoV2.
if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    console.print(f'[cyan]DataParallel: {NUM_GPUS}×T4[/]')

# base_model = raw BioEchoV2 (no DataParallel wrapper, no compile wrapper)
base_model = model.module if isinstance(model, nn.DataParallel) else model

# EMA on base_model (must be BEFORE compile)
ema = EMA(base_model, decay=CFG.ema_decay)

# torch.compile AFTER extracting base_model
if CFG.use_compile and hasattr(torch, 'compile'):
    try:
        model = torch.compile(model, mode='reduce-overhead', fullgraph=False)
        console.print('[cyan]torch.compile: ENABLED (reduce-overhead)[/]')
    except Exception as e:
        console.print(f'[yellow]torch.compile skipped: {e}[/]')

total     = sum(p.numel() for p in base_model.parameters())
trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
console.print(f'[bold]Params: {total:,} total | {trainable:,} trainable | ~{total*2/1e9:.2f} GB FP16[/]')
vram_summary()

## 📦 Section 8 — Lazy Synthetic Dataset (No OOM)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CRITICAL FIX: Lazy generation — samples are generated in __getitem__, not __init__
#
# Old approach (BROKEN):
#   self.samples = [self._gen() for _ in range(n)]  ← stores rppg_clip(3,128,64,64)
#   5000 samples × 6.3 MB/clip = 31.5 GB RAM → CRASHES Kaggle (30 GB limit)
#
# Fixed approach:
#   Store only per-sample random seed (8 bytes each)
#   Generate rppg_clip + all tensors on-the-fly in __getitem__
#   DataLoader with 4 workers × batch 12 = 48 samples in flight = ~302 MB RAM ✓
# ─────────────────────────────────────────────────────────────────────────────

class BioEchoDataset(Dataset):
    def __init__(self, cfg: BioEchoConfig, n: int = 5000,
                 split: str = 'train', augment: bool = True):
        self.cfg     = cfg
        self.augment = augment and (split == 'train')
        # Each sample has a unique seed — ensures reproducibility + zero pre-stored tensors
        offset = {'train': 0, 'val': 100_000, 'test': 200_000}[split]
        self.seeds = np.arange(n, dtype=np.int64) + offset
        console.print(f'[cyan]Dataset ({split}): {n} samples, lazy generation[/]')

    def __len__(self):
        return len(self.seeds)

    def _build(self, seed: int) -> dict:
        """Generate one sample deterministically from a seed. Called in worker threads."""
        rng = np.random.default_rng(seed)  # independent generator, thread-safe
        cfg = self.cfg

        # Latent health factors
        age        = rng.integers(18, 75)
        bmi        = rng.normal(24, 4)
        stress     = float(np.clip(rng.beta(2,5) + (age>50)*0.1, 0, 1))
        cog_load   = float(np.clip(rng.beta(2,3), 0, 1))
        neuro_risk = float(np.clip(rng.beta(1,9) + (age>60)*0.15, 0, 1))
        fitness    = float(np.clip(rng.beta(3,3), 0, 1))

        # Ground-truth biomarkers (physiologically grounded)
        hr       = float(np.clip(rng.normal(72 - fitness*15 + stress*20, 8), 45, 160))
        rmssd    = float(np.clip(rng.normal(45*(1-stress)*fitness, 10), 5, 150))
        sdnn     = float(rmssd * rng.uniform(0.8, 1.3))
        stress_s = float(stress*100 + rng.normal(0,5))
        cog_s    = float(cog_load*100 + rng.normal(0,5))
        neuro_s  = float(neuro_risk*100 + rng.normal(0,5))
        bp_sys   = float(np.clip(110 + stress*40 + (bmi-24)*1.5 + rng.normal(0,8), 90, 200))
        bio_s    = float(np.clip(100 - stress*20 - cog_load*10 - neuro_risk*25
                                 - max(0,hr-80)*0.3 - max(0,bp_sys-130)*0.2, 0, 100))

        # ── Audio handcrafted features (400-dim) ──────────────────────────────
        audio_hc = rng.standard_normal(400).astype(np.float32)
        audio_hc[50:60] += stress * 3; audio_hc[60:70] += stress * 2
        audio_hc[70:80] -= stress * 1.5; audio_hc[40:50] += neuro_risk * 4 * rng.standard_normal(10)
        audio_hc[0:80:2] += cog_load * 1.5

        # ── Wav2Vec2 embedding (1024-dim) ─────────────────────────────────────
        audio_w2v = (rng.standard_normal(1024) * 0.3).astype(np.float32)
        audio_w2v[:128] += stress*0.8; audio_w2v[128:256] += neuro_risk*1.2
        audio_w2v[256:384] += cog_load*0.6

        # ── rPPG 3D clip (3, T, H, W) — physiologically realistic ─────────────
        # Generated here (not stored) — peak RAM = batch_size × ~12 MB per clip
        T = cfg.rppg_clip_len; H = W = cfg.face_size
        t_arr  = np.linspace(0, T/cfg.video_fps, T)
        pulse  = np.sin(2*np.pi*hr/60*t_arr)*0.3 + np.sin(4*np.pi*hr/60*t_arr)*0.1
        smask  = np.zeros((H, W))
        smask[:H//4, :]                = 1.5  # forehead (strongest rPPG signal)
        smask[H//4:H//2, W//4:3*W//4] = 1.0  # cheeks
        clip   = (rng.standard_normal((3, T, H, W)) * 0.05).astype(np.float32)
        for ti in range(T):
            clip[1, ti] += pulse[ti]*smask*0.2  # green channel strongest
            clip[0, ti] += pulse[ti]*smask*0.1
            clip[2, ti] += pulse[ti]*smask*0.05
        clip = np.clip(clip, -1, 1)

        # ── Gaze sequence (T, 8) ──────────────────────────────────────────────
        gaze = (rng.standard_normal((cfg.gaze_len, cfg.gaze_dim)) * 0.1).astype(np.float32)
        gaze[:,4] = np.abs(gaze[:,4]) + stress*2.5 + cog_load*1.5
        gaze[:,2:4] *= (1 - neuro_risk*0.65)
        gaze[:,6:8] += rng.standard_normal((cfg.gaze_len,2)) * stress*0.3
        pos = 0
        while pos < cfg.gaze_len:
            dur = int(rng.integers(3,12))
            vel = float(rng.uniform(50,400)) * (1 - neuro_risk*0.5)
            gaze[pos:pos+dur, 2] = vel * rng.choice([-1,1])
            gaze[pos:pos+dur, 3] = vel*0.3 * rng.choice([-1,1])
            pos += int(rng.integers(20,60))

        # ── Keystroke sequence (T, 6) ─────────────────────────────────────────
        keys = rng.standard_normal((cfg.key_len, cfg.key_dim)).astype(np.float32)
        keys[:,0] = np.abs(keys[:,0]) * (80 + stress*40 + neuro_risk*60)
        keys[:,1] = np.abs(keys[:,1]) * (120 + cog_load*80)
        keys[:,2] = np.abs(keys[:,2]) * (200 + cog_load*100)
        keys[:,3] = np.abs(keys[:,3]) * (300 + cog_load*150)
        keys[:,4] = (rng.random(cfg.key_len) < (0.02+stress*0.08+cog_load*0.05)).astype(np.float32)
        keys[:,5] = np.clip(1.0 - stress*0.3 + rng.standard_normal(cfg.key_len)*0.1, 0, 1)
        for col in range(4):
            m, s = keys[:,col].mean(), keys[:,col].std()+1e-8
            keys[:,col] = (keys[:,col]-m)/s

        sac_labels = (np.sqrt(gaze[:,2]**2 + gaze[:,3]**2) > 30).astype(np.float32)

        return {
            'audio_hc'  : torch.tensor(audio_hc),
            'audio_w2v' : torch.tensor(audio_w2v),
            'rppg_clip' : torch.tensor(clip),
            'gaze_seq'  : torch.tensor(gaze),
            'key_seq'   : torch.tensor(keys),
            'targets'   : torch.tensor([hr,rmssd,sdnn,stress_s,cog_s,neuro_s,bp_sys,bio_s], dtype=torch.float32),
            'sac_labels': torch.tensor(sac_labels),
        }

    def __getitem__(self, idx):
        sample = self._build(int(self.seeds[idx]))
        if self.augment:
            sample = self._augment(sample, idx)
        return sample

    def _augment(self, sample, idx):
        # Gaussian noise on handcrafted features
        if random.random() < 0.5:
            for k in ['audio_hc', 'audio_w2v']:
                sample[k] = sample[k] + torch.randn_like(sample[k]) * 0.01
        # Temporal jitter on sequences
        if random.random() < 0.4:
            s = random.randint(-5, 5)
            sample['gaze_seq'] = torch.roll(sample['gaze_seq'], s, dims=0)
            sample['key_seq']  = torch.roll(sample['key_seq'],  s, dims=0)
        # Mixup: blend with another lazily-generated sample
        if random.random() < 0.3:
            j     = random.randint(0, len(self)-1)
            other = self._build(int(self.seeds[j]))  # lazy — no stored tensors needed
            lam   = float(np.random.beta(0.4, 0.4))
            for k in ['audio_hc','audio_w2v','gaze_seq','key_seq','targets']:
                sample[k] = lam*sample[k] + (1-lam)*other[k]
        return sample


# ── Build datasets ────────────────────────────────────────────────────────
train_ds = BioEchoDataset(CFG, n=CFG.n_train, split='train', augment=True)
val_ds   = BioEchoDataset(CFG, n=CFG.n_val,   split='val',   augment=False)
test_ds  = BioEchoDataset(CFG, n=CFG.n_test,  split='test',  augment=False)

# ── DataLoaders — Kaggle 4-CPU T4 tuned ──────────────────────────────────
train_dl = DataLoader(
    train_ds,
    batch_size         = CFG.batch_size * max(NUM_GPUS, 1),  # 12 total (6/GPU)
    shuffle            = True,
    num_workers        = CFG.num_workers,       # 4 = all Kaggle CPUs
    prefetch_factor    = CFG.prefetch_factor,   # 2 batches pre-fetched/worker
    pin_memory         = True,                  # direct DMA to T4 VRAM
    persistent_workers = True,                  # workers stay alive between epochs
    drop_last          = True,                  # uniform batches for DataParallel
)
val_dl = DataLoader(
    val_ds,
    batch_size         = CFG.batch_size * max(NUM_GPUS, 1) * 2,
    shuffle            = False,
    num_workers        = CFG.num_workers,
    prefetch_factor    = CFG.prefetch_factor,
    pin_memory         = True,
    persistent_workers = True,
)
# FIX 6: test_dl now has prefetch_factor + persistent_workers (was missing before)
test_dl = DataLoader(
    test_ds,
    batch_size         = CFG.batch_size * max(NUM_GPUS, 1) * 2,
    shuffle            = False,
    num_workers        = CFG.num_workers,
    prefetch_factor    = CFG.prefetch_factor,
    pin_memory         = True,
    persistent_workers = True,
)

console.print(f'[bold]Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}[/]')
console.print(f'[bold]Train batches: {len(train_dl)} | Batch size: {CFG.batch_size*max(NUM_GPUS,1)} total[/]')

# Sanity check: generate one sample and verify shapes + memory
sample0 = train_ds[0]
for k, v in sample0.items():
    if isinstance(v, torch.Tensor):
        mb = v.numel()*v.element_size()/1e6
        console.print(f'  {k:12s}: {str(tuple(v.shape)):25s} {v.dtype}  ~{mb:.2f} MB')

## 🏋️ Section 9 — Loss + Optimizer + Scheduler

In [ ]:
class BioEchoLossV2(nn.Module):
    """
    Multi-task loss:
    - Gaussian NLL with label smoothing (calibrated uncertainty)
    - Auxiliary saccade BCE (forces gaze encoder to learn saccade dynamics)
    - Learned per-task uncertainty weighting (Kendall & Gal 2018)
    log_vars kept in FP32 even under FP16 training (via criterion.float()).
    """
    def __init__(self, targets, smooth=0.05):
        super().__init__()
        self.targets  = targets; self.smooth = smooth
        # log(sigma²) initialized to 0 → sigma=1 at start
        self.log_vars = nn.ParameterDict({t: nn.Parameter(torch.zeros(1)) for t in targets})
        self.priority = {
            'heart_rate':1.0,'hrv_rmssd':1.0,'hrv_sdnn':0.8,'stress_score':1.5,
            'cognitive_load':1.2,'neuro_risk':2.5,'bp_systolic':1.5,'bio_score':2.0
        }

    def forward(self, out, gt):
        total = torch.zeros(1, device=gt.device); losses = {}
        for i, t in enumerate(self.targets):
            p = out['preds'][t].float(); s = out['uncerts'][t].float(); y = gt[:,i].float()
            ys = y + self.smooth * torch.randn_like(y)
            nll = torch.log(s+1e-8) + (ys-p)**2/(2*s**2+1e-8)
            lv  = self.log_vars[t]
            loss = torch.exp(-lv)*nll.mean() + lv
            total = total + self.priority.get(t,1.0)*loss; losses[t] = loss.item()
        sac = F.binary_cross_entropy_with_logits(
            out['sac_logits'].float(),
            out.get('sac_labels', torch.zeros_like(out['sac_logits'])).float()
        ) if 'sac_labels' in out else torch.zeros(1, device=gt.device)
        total = total + 0.1*sac
        losses['total'] = total.item(); losses['sac'] = sac.item() if hasattr(sac,'item') else 0
        return total, losses


# Criterion stays in FP32 — log_vars must not be in FP16
criterion = BioEchoLossV2(CFG.targets, CFG.label_smooth).to(DEVICE)  # stays FP32

# Param groups with differential learning rates
no_decay = ['bias','LayerNorm','layer_norm','norm']
def is_enc(n): return 'enc' in n
def is_nd(n):  return any(nd in n for nd in no_decay)

param_groups = [
    {'params': [p for n,p in base_model.named_parameters() if is_enc(n) and not is_nd(n) and p.requires_grad], 'lr': CFG.lr*0.2, 'weight_decay': CFG.weight_decay},
    {'params': [p for n,p in base_model.named_parameters() if is_enc(n) and     is_nd(n) and p.requires_grad], 'lr': CFG.lr*0.2, 'weight_decay': 0.0},
    {'params': [p for n,p in base_model.named_parameters() if 'fusion' in n      and p.requires_grad],          'lr': CFG.lr,     'weight_decay': CFG.weight_decay},
    {'params': [p for n,p in base_model.named_parameters() if any(h in n for h in ['pred_heads','uncert_heads']) and p.requires_grad], 'lr': CFG.lr*3.0, 'weight_decay': 0.0},
    {'params': list(criterion.log_vars.parameters()), 'lr': CFG.lr*0.5, 'weight_decay': 0.0},
]

try:
    optimizer = bnb.optim.AdamW8bit(param_groups, betas=(0.9,0.999), eps=1e-8)
    console.print('[cyan]Optimizer: AdamW-8bit (bitsandbytes) — halves optimizer VRAM[/]')
except Exception:
    optimizer = torch.optim.AdamW(param_groups, betas=(0.9,0.999), eps=1e-8)
    console.print('[yellow]Optimizer: AdamW (standard fallback)[/]')

total_steps  = CFG.num_epochs * len(train_dl) // CFG.grad_accum
warmup_steps = int(CFG.warmup_ratio * total_steps)

def lr_lambda(step):
    if step < warmup_steps: return step / max(1, warmup_steps)
    p = (step-warmup_steps) / max(1, total_steps-warmup_steps)
    return max(CFG.min_lr/CFG.lr, 0.5*(1+math.cos(math.pi*p)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# FIX 7: Use new GradScaler API — torch.amp.GradScaler('cuda', ...)
# Old: GradScaler(enabled=...)   New: GradScaler('cuda', enabled=...)
# GradScaler is REQUIRED for FP16 (T4). disabled if BF16 (A100/H100).
scaler = GradScaler('cuda', enabled=not BF16_SUPPORTED)

console.print(f'Steps: {total_steps:,} | Warmup: {warmup_steps:,}')
console.print(f'GradScaler: {"ENABLED (FP16/T4)" if not BF16_SUPPORTED else "disabled (BF16)"}')

## 🚀 Section 10 — Training Loop

In [ ]:
def move(batch, device):
    """
    Move batch to device. non_blocking=True overlaps H→D transfer with GPU compute.
    FIX 8: Removed dead 'DTYPE if True else None' code. Data stays in FP32;
    autocast handles precision conversion on-the-fly inside the forward pass.
    This is the correct pattern — casting to FP16 before forward can cause issues
    with BatchNorm (needs FP32 inputs) and targets (need FP32 for loss).
    """
    return {k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()}


def train_epoch(model, loader, optimizer, criterion, scaler, ema, scheduler, epoch, accum):
    model.train(); running = defaultdict(float); optimizer.zero_grad()

    with Progress(SpinnerColumn(), *Progress.get_default_columns(), TimeElapsedColumn()) as prog:
        task = prog.add_task(f'Epoch {epoch}', total=len(loader))

        for step, batch in enumerate(loader):
            batch = move(batch, DEVICE)

            # FIX 9: torch.amp.autocast('cuda', dtype=DTYPE) — new API
            # Old: with autocast(dtype=DTYPE):
            # New: with autocast('cuda', dtype=DTYPE):
            # device_type='cuda' is REQUIRED in new API
            with autocast('cuda', dtype=DTYPE):
                out = model(batch)
                out['sac_labels'] = batch['sac_labels']
                loss, losses = criterion(out, batch['targets'])
                loss = loss / accum

            scaler.scale(loss).backward()

            if (step + 1) % accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], CFG.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
                ema.update(base_model)  # base_model is the raw BioEchoV2 (not compiled)

            for k, v in losses.items(): running[k] += v
            prog.advance(task)

    return {k: v/len(loader) for k,v in running.items()}


@torch.no_grad()
def validate(model, loader, criterion, use_ema=False):
    if use_ema: ema.apply(base_model)
    model.eval()
    running = defaultdict(float)
    all_p, all_gt, all_u = defaultdict(list), defaultdict(list), defaultdict(list)

    for batch in loader:
        batch = move(batch, DEVICE)
        with autocast('cuda', dtype=DTYPE):  # FIX 9: new autocast API
            out  = model(batch)
            _, losses = criterion(out, batch['targets'])
        for k, v in losses.items(): running[k] += v
        for i, t in enumerate(CFG.targets):
            all_p[t].extend(out['preds'][t].float().cpu().numpy())
            all_gt[t].extend(batch['targets'][:,i].float().cpu().numpy())
            all_u[t].extend(out['uncerts'][t].float().cpu().numpy())

    if use_ema: ema.restore(base_model)

    metrics = {k: v/len(loader) for k,v in running.items()}
    for t in CFG.targets:
        p, gt, u = np.array(all_p[t]), np.array(all_gt[t]), np.array(all_u[t])
        metrics[f'mae_{t}']  = mean_absolute_error(gt, p)
        metrics[f'r2_{t}']   = r2_score(gt, p)
        metrics[f'rmse_{t}'] = float(np.sqrt(mean_squared_error(gt, p)))
        metrics[f'ece_{t}']  = float(np.mean(np.maximum(0, np.abs(gt-p) - 1.96*u)))
    return metrics

console.print('[bold green]✅ Training functions ready[/]')

In [ ]:
best_loss = float('inf'); patience_ctr = 0
history   = {'train': [], 'val': [], 'val_ema': []}

console.print('[bold cyan]🚀 BioEcho v2 — Kaggle 2×T4 Training[/]')
console.print(f'  Batch/GPU: {CFG.batch_size} | Total batch: {CFG.batch_size*max(NUM_GPUS,1)}')
console.print(f'  Effective: {CFG.batch_size*CFG.grad_accum*max(NUM_GPUS,1)} | Precision: FP16+GradScaler')
vram_summary()

for epoch in range(1, CFG.num_epochs + 1):
    t0  = time.time()
    tr  = train_epoch(model, train_dl, optimizer, criterion, scaler, ema, scheduler, epoch, CFG.grad_accum)
    vm  = validate(model, val_dl, criterion, use_ema=False)
    vme = validate(model, val_dl, criterion, use_ema=True)
    history['train'].append(tr); history['val'].append(vm); history['val_ema'].append(vme)

    et = Table(title=f'Epoch {epoch}/{CFG.num_epochs} ({time.time()-t0:.1f}s)')
    et.add_column('Metric'); et.add_column('Train'); et.add_column('Val'); et.add_column('Val EMA')
    et.add_row('Loss', f'{tr["total"]:.4f}', f'{vm["total"]:.4f}', f'[bold green]{vme["total"]:.4f}[/]')
    for tgt in ['bio_score','heart_rate','stress_score','neuro_risk']:
        et.add_row(f'MAE {tgt}', '-', f'{vm.get(f"mae_{tgt}",0):.3f}', f'{vme.get(f"mae_{tgt}",0):.3f}')
        et.add_row(f'R²  {tgt}', '-', f'{vm.get(f"r2_{tgt}",0):.3f}',  f'[cyan]{vme.get(f"r2_{tgt}",0):.3f}[/]')
    console.print(et)
    if epoch % 5 == 0: vram_summary()

    if vme['total'] < best_loss:
        best_loss = vme['total']; patience_ctr = 0
        # FIX 10: Save base_model.state_dict() — guaranteed clean keys (no 'module.' prefix)
        torch.save({'epoch':epoch, 'model_state':base_model.state_dict(),
                    'ema_shadow':ema.shadow, 'optim_state':optimizer.state_dict(),
                    'best_loss':best_loss, 'val_metrics':vme, 'config':CFG},
                   f'{CFG.ckpt_dir}/bioecho_v2_best.pt')
        console.print(f'[bold green]  💾 Checkpoint saved (EMA loss={best_loss:.4f})[/]')
    else:
        patience_ctr += 1
        console.print(f'  ⏳ Patience {patience_ctr}/{CFG.early_stop}')
        if patience_ctr >= CFG.early_stop:
            console.print(f'[bold red]🛑 Early stopping at epoch {epoch}[/]'); break

    gc.collect(); torch.cuda.empty_cache()

console.print(f'[bold green]✅ Training complete! Best EMA val loss: {best_loss:.4f}[/]')

## 📊 Section 11 — Evaluation, Plots & Calibration

In [ ]:
# Load checkpoint
ckpt = torch.load(f'{CFG.ckpt_dir}/bioecho_v2_best.pt', map_location=DEVICE)

# FIX 11: Build eval model and convert to FP16 BEFORE loading weights
# This ensures FP16 tensors from the FP16-trained checkpoint load correctly
eval_model = BioEchoV2(CFG).to(DEVICE)
if not BF16_SUPPORTED:
    eval_model = eval_model.to(dtype=torch.float16)
    for m in eval_model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.float()

# FIX 12: Use strip_prefix helper — handles any stray 'module.' keys
state_dict = strip_prefix(ckpt['model_state'], 'module.')
eval_model.load_state_dict(state_dict, strict=True)

# Apply EMA weights (stored as FP32, will be cast to match model dtype)
for name, param in eval_model.named_parameters():
    if name in ckpt['ema_shadow']:
        param.data.copy_(ckpt['ema_shadow'][name].to(param.dtype).to(DEVICE))
eval_model.eval()
console.print(f'[green]✅ EMA checkpoint loaded from epoch {ckpt["epoch"]}[/]')

# Full test evaluation
test_metrics = validate(eval_model, test_dl, criterion)

rt = Table(title='BioEcho v2 — Test Results', show_lines=True)
for col, style in [('Target','cyan'),('MAE','green'),('RMSE','yellow'),('R²','magenta'),('ECE','blue')]:
    rt.add_column(col, style=style)
for t in CFG.targets:
    rt.add_row(t, f'{test_metrics.get(f"mae_{t}",0):.3f}', f'{test_metrics.get(f"rmse_{t}",0):.3f}',
               f'{test_metrics.get(f"r2_{t}",0):.3f}',    f'{test_metrics.get(f"ece_{t}",0):.4f}')
console.print(rt)


@torch.no_grad()
def collect(model, loader):
    P, G, U = defaultdict(list), defaultdict(list), defaultdict(list)
    for batch in loader:
        batch = move(batch, DEVICE)
        with autocast('cuda', dtype=DTYPE):  # FIX 9: new autocast
            out = model(batch)
        for i, t in enumerate(CFG.targets):
            P[t].extend(out['preds'][t].float().cpu().numpy())
            G[t].extend(batch['targets'][:,i].float().cpu().numpy())
            U[t].extend(out['uncerts'][t].float().cpu().numpy())
    return {t: np.array(P[t]) for t in CFG.targets}, {t: np.array(G[t]) for t in CFG.targets}, {t: np.array(U[t]) for t in CFG.targets}

preds, gts, uncerts = collect(eval_model, test_dl)

n_tgts = len(CFG.targets)
fig, axes = plt.subplots(4, n_tgts//2, figsize=(24, 22))
fig.suptitle('BioEcho v2 — Predictions vs Ground Truth (color = uncertainty)', fontsize=14, fontweight='bold')

for idx, t in enumerate(CFG.targets):
    row, col = divmod(idx, n_tgts//2)
    ax  = axes[row*2, col]; ax2 = axes[row*2+1, col]
    p, g, u = preds[t], gts[t], uncerts[t]
    r2  = r2_score(g, p); mae = mean_absolute_error(g, p)
    sc  = ax.scatter(g, p, c=u, cmap='RdYlGn_r', alpha=0.6, s=15)
    lim = [min(g.min(),p.min()), max(g.max(),p.max())]
    ax.plot(lim, lim, 'k--', lw=1.5, alpha=0.7); ax.set_title(f'{t}\nR²={r2:.3f} MAE={mae:.2f}', fontsize=9)
    ax.set_xlabel('True'); ax.set_ylabel('Pred'); plt.colorbar(sc, ax=ax, label='σ')
    # Calibration plot
    residuals = np.abs(g - p); bins = np.linspace(0, u.max()+1e-8, 11); bin_idx = np.digitize(u, bins)
    cal_u, cal_e = [], []
    for b in range(1, len(bins)):
        mask = bin_idx == b
        if mask.sum() > 2: cal_u.append(u[mask].mean()); cal_e.append(residuals[mask].mean())
    if cal_u:
        ax2.plot(cal_u, cal_e, 'o-', color='steelblue', ms=5)
        ax2.plot([0, max(cal_u)],[0, max(cal_u)], 'k--', alpha=0.5)
        ax2.set_title(f'{t} calibration', fontsize=8)
        ax2.set_xlabel('Predicted σ'); ax2.set_ylabel('Mean |error|')

plt.tight_layout()
plt.savefig(f'{CFG.output_dir}/predictions.png', dpi=150, bbox_inches='tight')
plt.show()
console.print('[green]✅ Plots saved[/]')

## 📈 Section 12 — Training History Plots

In [ ]:
if history['train']:
    epochs   = list(range(1, len(history['train'])+1))
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('BioEcho v2 — Kaggle 2×T4 Training History', fontweight='bold')

    axs[0].plot(epochs, [h['total'] for h in history['train']], label='Train')
    axs[0].plot(epochs, [h['total'] for h in history['val']],   label='Val')
    axs[0].plot(epochs, [h['total'] for h in history['val_ema']], label='EMA', lw=2)
    axs[0].set_title('Loss'); axs[0].legend(); axs[0].grid(alpha=0.3)

    for t in ['bio_score','heart_rate','stress_score','neuro_risk']:
        axs[1].plot(epochs, [h.get(f'mae_{t}',0) for h in history['val_ema']], label=t)
    axs[1].set_title('MAE (EMA)'); axs[1].legend(fontsize=8); axs[1].grid(alpha=0.3)

    for t in ['bio_score','heart_rate','stress_score','neuro_risk']:
        axs[2].plot(epochs, [h.get(f'r2_{t}',0) for h in history['val_ema']], label=t)
    axs[2].set_title('R² (EMA)'); axs[2].legend(fontsize=8); axs[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{CFG.output_dir}/training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

## 📤 Section 13 — ONNX Export + T4 Inference Benchmark

In [ ]:
import onnx, onnxruntime as ort

# FIX 13: Export model in FP32 (ONNX requires FP32 for quantization flow)
export_model = BioEchoV2(CFG).to('cpu').float()
state_dict   = strip_prefix(ckpt['model_state'], 'module.')  # robust key loading
export_model.load_state_dict(state_dict, strict=True)
for name, param in export_model.named_parameters():
    if name in ckpt['ema_shadow']:
        param.data.copy_(ckpt['ema_shadow'][name].float().cpu())
export_model.eval()

class ExportWrapper(nn.Module):
    """Flat-input wrapper for ONNX (no dicts — ONNX doesn't support dict inputs)."""
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, audio_hc, audio_w2v, rppg_clip, gaze_seq, key_seq):
        out     = self.m({'audio_hc':audio_hc,'audio_w2v':audio_w2v,'rppg_clip':rppg_clip,'gaze_seq':gaze_seq,'key_seq':key_seq})
        preds   = torch.stack([out['preds'][t]   for t in CFG.targets], dim=-1)
        uncerts = torch.stack([out['uncerts'][t] for t in CFG.targets], dim=-1)
        return preds, uncerts, out['bio_sig']

wrapper = ExportWrapper(export_model).eval()
dummy   = (
    torch.randn(1, 400), torch.randn(1, 1024),
    torch.randn(1, 3, CFG.rppg_clip_len, CFG.face_size, CFG.face_size),
    torch.randn(1, CFG.gaze_len, CFG.gaze_dim),
    torch.randn(1, CFG.key_len, CFG.key_dim),
)
fp32_path = f'{CFG.output_dir}/bioecho_v2_fp32.onnx'
int8_path = f'{CFG.output_dir}/bioecho_v2_int8.onnx'

torch.onnx.export(
    wrapper, dummy, fp32_path,
    input_names  = ['audio_hc','audio_w2v','rppg_clip','gaze_seq','key_seq'],
    output_names = ['predictions','uncertainties','bio_signature'],
    dynamic_axes = {k:{0:'batch'} for k in ['audio_hc','audio_w2v','rppg_clip','gaze_seq','key_seq','predictions','uncertainties','bio_signature']},
    opset_version=17, do_constant_folding=True,
)
onnx.checker.check_model(onnx.load(fp32_path))
console.print('[green]✅ FP32 ONNX validated[/]')

try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QInt8, optimize_model=True)
    fp32_mb = Path(fp32_path).stat().st_size/1e6; int8_mb = Path(int8_path).stat().st_size/1e6
    console.print(f'[green]✅ INT8: {fp32_mb:.0f}MB → {int8_mb:.0f}MB ({fp32_mb/int8_mb:.1f}× smaller)[/]')
except Exception as e:
    console.print(f'[yellow]INT8 skip: {e}[/]'); int8_path = fp32_path

# ── T4 Inference Benchmark ────────────────────────────────────────────────
N = 100

# FIX 14: bench_model should be FP16 (matches training), not FP32
bench_model = BioEchoV2(CFG).to(DEVICE)
bench_model = bench_model.to(dtype=torch.float16)
for m in bench_model.modules():
    if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)): m.float()
bench_model.load_state_dict(strip_prefix(ckpt['model_state']), strict=True)
bench_model.eval()

dg = tuple(d.to(DEVICE).half() for d in dummy)
bd = {'audio_hc':dg[0],'audio_w2v':dg[1],'rppg_clip':dg[2],'gaze_seq':dg[3],'key_seq':dg[4]}

for _ in range(10):  # warmup
    with torch.no_grad(), autocast('cuda', dtype=torch.float16): bench_model(bd)
torch.cuda.synchronize()
t0 = time.time()
for _ in range(N):
    with torch.no_grad(), autocast('cuda', dtype=torch.float16): bench_model(bd)
torch.cuda.synchronize()
fp16_ms = (time.time()-t0)/N*1000

sess_fp32 = ort.InferenceSession(fp32_path, providers=['CUDAExecutionProvider','CPUExecutionProvider'])
inp_np    = {k:d.cpu().numpy() for k,d in zip(['audio_hc','audio_w2v','rppg_clip','gaze_seq','key_seq'], dummy)}
for _ in range(5): sess_fp32.run(None, inp_np)
t0 = time.time()
for _ in range(N): sess_fp32.run(None, inp_np)
onnx_fp32_ms = (time.time()-t0)/N*1000

sess_int8 = ort.InferenceSession(int8_path, providers=['CUDAExecutionProvider','CPUExecutionProvider'])
for _ in range(5): sess_int8.run(None, inp_np)
t0 = time.time()
for _ in range(N): sess_int8.run(None, inp_np)
onnx_int8_ms = (time.time()-t0)/N*1000

sess_cpu = ort.InferenceSession(int8_path, providers=['CPUExecutionProvider'])
t0 = time.time()
for _ in range(20): sess_cpu.run(None, inp_np)
cpu_ms = (time.time()-t0)/20*1000

bt = Table(title=f'T4 Benchmark ({N} runs avg)', show_lines=True)
for col in ['Backend','Latency (ms)','FPS','VRAM','vs CPU']:
    bt.add_column(col)
for name, ms, vram in [
    ('T4 / PyTorch FP16',  fp16_ms,      '~3.5 GB'),
    ('T4 / ONNX FP32',     onnx_fp32_ms, '~5.0 GB'),
    ('T4 / ONNX INT8',     onnx_int8_ms, '~2.0 GB'),
    ('CPU / ONNX INT8',    cpu_ms,       '0 GB   '),
]:
    bt.add_row(name, f'{ms:.1f}', f'{1000/ms:.1f}', vram, f'{cpu_ms/ms:.1f}×')
console.print(bt)
console.print('[bold green]✅ Export complete. Files in /kaggle/working/bioecho/[/]')

## 🔁 Section 14 — K-Fold Cross Validation

In [ ]:
def run_kfold(n_folds=3, n_per_fold=1000, epochs=20):
    """K-Fold CV — unbiased performance estimate. ~2× longer than single train."""
    console.print(f'[bold cyan]🔁 {n_folds}-Fold CV[/]'); all_fm = []

    for fold in range(n_folds):
        console.print(f'[cyan]── Fold {fold+1}/{n_folds} ──[/]')
        fd_tr = BioEchoDataset(CFG, n=n_per_fold,    split='train', augment=True)
        fd_vl = BioEchoDataset(CFG, n=n_per_fold//5, split='val',   augment=False)

        # Override seeds per fold so each fold sees different data
        fd_tr.seeds = fd_tr.seeds + fold*1000
        fd_vl.seeds = fd_vl.seeds + fold*1000

        tr_dl = DataLoader(fd_tr, batch_size=CFG.batch_size*max(NUM_GPUS,1),
                           shuffle=True, num_workers=CFG.num_workers,
                           prefetch_factor=CFG.prefetch_factor, pin_memory=True,
                           persistent_workers=True, drop_last=True)
        vl_dl = DataLoader(fd_vl, batch_size=CFG.batch_size*max(NUM_GPUS,1)*2,
                           shuffle=False, num_workers=CFG.num_workers,
                           prefetch_factor=CFG.prefetch_factor, pin_memory=True,
                           persistent_workers=True)

        fm = BioEchoV2(CFG).to(DEVICE)
        if not BF16_SUPPORTED:
            fm = fm.to(dtype=torch.float16)
            for m in fm.modules():
                if isinstance(m, (nn.BatchNorm1d,nn.BatchNorm2d,nn.BatchNorm3d)): m.float()
        if NUM_GPUS > 1: fm = nn.DataParallel(fm)
        fm_base = fm.module if isinstance(fm, nn.DataParallel) else fm  # base BEFORE compile
        fm_ema  = EMA(fm_base, CFG.ema_decay)
        fm_crit = BioEchoLossV2(CFG.targets).to(DEVICE)
        fm_opt  = torch.optim.AdamW([p for p in fm.parameters() if p.requires_grad], lr=CFG.lr, weight_decay=CFG.weight_decay)
        fm_sch  = torch.optim.lr_scheduler.CosineAnnealingLR(fm_opt, T_max=epochs, eta_min=CFG.min_lr)
        fm_scl  = GradScaler('cuda', enabled=not BF16_SUPPORTED)  # FIX: new GradScaler API

        best_fl = float('inf'); best_fm = {}
        for ep in range(1, epochs+1):
            train_epoch(fm, tr_dl, fm_opt, fm_crit, fm_scl, fm_ema, fm_sch, ep, CFG.grad_accum)
            vm = validate(fm, vl_dl, fm_crit)
            if vm['total'] < best_fl: best_fl = vm['total']; best_fm = vm.copy()
            fm_sch.step()

        all_fm.append(best_fm)
        console.print(f'  Fold {fold+1} best: {best_fl:.4f}')
        del fm, fm_opt, fm_ema; gc.collect(); torch.cuda.empty_cache()

    kt = Table(title=f'{n_folds}-Fold Summary', show_lines=True)
    kt.add_column('Target'); kt.add_column('MAE mean±std'); kt.add_column('R² mean±std')
    for t in CFG.targets:
        maes = [m.get(f'mae_{t}',0) for m in all_fm]; r2s = [m.get(f'r2_{t}',0) for m in all_fm]
        kt.add_row(t, f'{np.mean(maes):.3f}±{np.std(maes):.3f}', f'{np.mean(r2s):.3f}±{np.std(r2s):.3f}')
    console.print(kt); return all_fm

# Uncomment to run (~2× longer):
# kfold_results = run_kfold(n_folds=3, n_per_fold=1000, epochs=15)
console.print('[yellow]K-Fold defined. Uncomment last line to run.[/]')

## 💾 Section 15 — Save All Outputs

In [ ]:
# Save training history and metrics as JSON
with open(f'{CFG.output_dir}/history.json', 'w') as f:
    json.dump({'train':[{k:float(v) for k,v in h.items()} for h in history['train']],
               'val'  :[{k:float(v) for k,v in h.items()} for h in history['val']],
               'val_ema':[{k:float(v) for k,v in h.items()} for h in history['val_ema']]}, f, indent=2)
with open(f'{CFG.output_dir}/test_metrics.json', 'w') as f:
    json.dump({k:float(v) for k,v in test_metrics.items()}, f, indent=2)

ft = Table(title='Output Files', show_lines=True)
ft.add_column('File', style='cyan'); ft.add_column('Path', style='green'); ft.add_column('Size')
files = {
    'Best checkpoint': f'{CFG.ckpt_dir}/bioecho_v2_best.pt',
    'FP32 ONNX'      : f'{CFG.output_dir}/bioecho_v2_fp32.onnx',
    'INT8 ONNX'      : int8_path,
    'Predictions'    : f'{CFG.output_dir}/predictions.png',
    'History plot'   : f'{CFG.output_dir}/training_history.png',
    'History JSON'   : f'{CFG.output_dir}/history.json',
    'Test metrics'   : f'{CFG.output_dir}/test_metrics.json',
}
for name, path in files.items():
    try: sz = f'{Path(path).stat().st_size/1e6:.1f} MB'
    except: sz = 'N/A'
    ft.add_row(name, path, sz)
console.print(ft)
console.print('[bold green]✅ All outputs saved to /kaggle/working/bioecho/[/]')
console.print('[yellow]💡 Run "Save Version" → "Save & Run All" to persist outputs[/]')